In [ ]:
%run "./02_EMS_preprocessing.ipynb"

ROOT_DIR  : C:\Users\Admin\Desktop\Projet_Artemis2
DATA_FILE : C:\Users\Admin\Desktop\Projet_Artemis2\data\Artemis.csv | existe: True
RANDOM_SEED: 42 | DEVICE: cuda
CONFIGURATION DU PROJET HESS

BATTERIE ÉNERGIE
Énergie          : 13709.89 Wh
Tension          : 450.00 V
Capacité         : 30.4664 Ah
Masse            : 55.12 kg
Courant recharge : -14.00 A
Courant décharge : 28.00 A
Configuration    : 125S7P

BATTERIE PUISSANCE
Énergie          : 2987.12 Wh
Tension          : 402.60 V
Capacité         : 7.4196 Ah
Masse            : 50.02 kg
Courant recharge : -130.00 A
Courant décharge : 400.00 A
Configuration    : 122S2P

HESS
Énergie totale   : 16697.01 Wh
Masse totale     : 105.14 kg
Tension du bus   : 402.60 V
Puissance min    : -58638.00 W
Puissance max    : 173640.00 W

MODÈLES
LSTM seul        : 7 → 64 → 3
LSTM NS          : 15 → 64 → 3
GNN simple       : 12 → 32 → 1
MLP simple       : 5 → 64 → 32 → 1
MLP NS           : 17 → 64 → 32 → 1

Device           : cuda
Fichier          : 

In [29]:
# Cellule 0 : chargement


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

files = {
    "train": PROCESSED_DIR / "train_with_symbolic_states.csv",
    "validation": PROCESSED_DIR / "validation_with_symbolic_states.csv",
    "test": PROCESSED_DIR / "test_with_symbolic_states.csv",
}

for path in files.values():
    if not path.exists():
        raise FileNotFoundError(f"Fichier absent : {path}")

train_df = pd.read_csv(files["train"])
val_df = pd.read_csv(files["validation"])
test_df = pd.read_csv(files["test"])

LSTM_NS_CHECKPOINT = globals().get("LSTM_NS_CHECKPOINT", CHECKPOINTS_DIR / "EMS_LSTM_neurosymbolic.pt")
LSTM_NS_SCALER_FILE = globals().get("LSTM_NS_SCALER_FILE", MODELS_DIR / "EMS_LSTM_neurosymbolic_scalers.npz")
SOC_TOL = globals().get("SOC_NUMERICAL_TOLERANCE", globals().get("SOC_TOLERANCE", 5e-4))

if DT_SECONDS is None:
    DT_SECONDS = float(train_df[TIME_COL].diff().dropna().median())

print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape)
print("DT         :", DT_SECONDS)

Train      : (7265, 39)
Validation : (3632, 39)
Test       : (3633, 39)
DT         : 1.0


Sélection des entrées symboliques

In [30]:
# Cellule 2 : selection des entrees symboliques
required_cols = (
    LSTM_INPUT_COLS + SYMBOLIC_COLS
    + [TIME_COL, "I_EB", "I_PB", "SOC_EB", "SOC_PB", "hasPower"]
)

for name, data in {"train": train_df, "validation": val_df, "test": test_df}.items():
    missing = sorted(set(required_cols) - set(data.columns))
    if missing:
        raise ValueError(f"{name} : colonnes absentes {missing}")

symbolic_nunique = train_df[SYMBOLIC_COLS].nunique()
CONSTANT_SYMBOLIC_COLS = symbolic_nunique[symbolic_nunique <= 1].index.tolist()
SYMBOLIC_MODEL_COLS = [col for col in SYMBOLIC_COLS if col not in CONSTANT_SYMBOLIC_COLS]
LSTM_NS_MODEL_INPUT_COLS = LSTM_INPUT_COLS + SYMBOLIC_MODEL_COLS

print("Indicateurs constants :", CONSTANT_SYMBOLIC_COLS)
print("Indicateurs utilisés  :", SYMBOLIC_MODEL_COLS)
print("Nombre d'entrées      :", len(LSTM_NS_MODEL_INPUT_COLS))

Indicateurs constants : ['EB_available', 'PB_available', 'EB_low_SOC', 'PB_low_SOC']
Indicateurs utilisés  : ['high_power_demand', 'regenerative_braking', 'zero_power_demand', 'converter_risk']
Nombre d'entrées      : 11


Construction des fenêtres

In [31]:
# Cellule 4 : fonction de construction des fenetres
def build_windows_ns(data, cols, window, horizon):
    x = data[cols].to_numpy(dtype=np.float32)
    p_dem = data["hasPower"].to_numpy(dtype=np.float32)
    soc_eb = data["SOC_EB"].to_numpy(dtype=np.float32)
    soc_pb = data["SOC_PB"].to_numpy(dtype=np.float32)
    i_eb = data["I_EB"].to_numpy(dtype=np.float32)
    i_pb = data["I_PB"].to_numpy(dtype=np.float32)
    time = data[TIME_COL].to_numpy()

    anchors = np.arange(window - 1, len(data) - horizon)
    targets = anchors + horizon

    X = np.empty((len(anchors), window, len(cols)), dtype=np.float32)
    y = np.empty((len(anchors), 3), dtype=np.float32)
    aux = np.empty((len(anchors), 5), dtype=np.float32)

    for k, anchor in enumerate(anchors):
        start = anchor - window + 1
        target = targets[k]

        X[k] = x[start:anchor + 1]
        y[k, 0] = p_dem[target]
        y[k, 1] = soc_eb[target] - soc_eb[anchor]
        y[k, 2] = soc_pb[target] - soc_pb[anchor]

        dsoc_eb_phys = -np.sum(i_eb[anchor:target], dtype=np.float64) * DT_SECONDS / (3600.0 * CAPACITY_EB_AH)
        dsoc_pb_phys = -np.sum(i_pb[anchor:target], dtype=np.float64) * DT_SECONDS / (3600.0 * CAPACITY_PB_AH)

        protected_discharge = (soc_eb[anchor] <= SOC_EB_MIN and p_dem[anchor] > EPS_POWER_W)

        aux[k, 0] = dsoc_eb_phys
        aux[k, 1] = dsoc_pb_phys
        aux[k, 2] = float(protected_discharge)
        aux[k, 3] = soc_eb[anchor]
        aux[k, 4] = soc_pb[anchor]

    metadata = pd.DataFrame({
        "anchor_index": anchors, "target_index": targets,
        "time_anchor": time[anchors], "time_target": time[targets],
        "SOC_EB_anchor": soc_eb[anchors], "SOC_PB_anchor": soc_pb[anchors],
        "SOC_EB_target": soc_eb[targets], "SOC_PB_target": soc_pb[targets],
    })
    return X, y, aux, metadata

Création des fenêtres

In [32]:
# Cellule 6 : creation des fenetres
X_train, y_train, aux_train, meta_train = build_windows_ns(train_df, LSTM_NS_MODEL_INPUT_COLS, WINDOW, HORIZON)
X_val, y_val, aux_val, meta_val = build_windows_ns(val_df, LSTM_NS_MODEL_INPUT_COLS, WINDOW, HORIZON)
X_test, y_test, aux_test, meta_test = build_windows_ns(test_df, LSTM_NS_MODEL_INPUT_COLS, WINDOW, HORIZON)

print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)
print("Règle active dans le train :", int(aux_train[:, 2].sum()))
print("Règle active dans le test :", int(aux_test[:, 2].sum()))

Train      : (7241, 20, 11) (7241, 3)
Validation : (3608, 20, 11) (3608, 3)
Test       : (3609, 20, 11) (3609, 3)
Règle active dans le train : 0
Règle active dans le test : 374


Normalisation

In [33]:
# Cellule 6
X_train, y_train, aux_train, meta_train = build_windows_ns(train_df, LSTM_NS_MODEL_INPUT_COLS, WINDOW, HORIZON)
X_val, y_val, aux_val, meta_val = build_windows_ns(val_df, LSTM_NS_MODEL_INPUT_COLS, WINDOW, HORIZON)
X_test, y_test, aux_test, meta_test = build_windows_ns(test_df, LSTM_NS_MODEL_INPUT_COLS, WINDOW, HORIZON)

print("Train      :", X_train.shape, y_train.shape)
print("Validation :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)
print("Règle active dans le train :", int(aux_train[:, 2].sum()))
print("Règle active dans le test :", int(aux_test[:, 2].sum()))

Train      : (7241, 20, 11) (7241, 3)
Validation : (3608, 20, 11) (3608, 3)
Test       : (3609, 20, 11) (3609, 3)
Règle active dans le train : 0
Règle active dans le test : 374


In [34]:
# Cellule 8 : normalisation
x_train_numeric = train_df[LSTM_INPUT_COLS].to_numpy(dtype=np.float32)
x_mean_numeric = x_train_numeric.mean(axis=0, dtype=np.float64).astype(np.float32)
x_std_numeric = x_train_numeric.std(axis=0, dtype=np.float64).astype(np.float32)

if np.any(x_std_numeric < 1e-12):
    cols = np.array(LSTM_INPUT_COLS)[x_std_numeric < 1e-12]
    raise ValueError(f"Variables constantes : {cols.tolist()}")

x_mean = np.concatenate([x_mean_numeric, np.zeros(len(SYMBOLIC_MODEL_COLS), dtype=np.float32)])
x_std = np.concatenate([x_std_numeric, np.ones(len(SYMBOLIC_MODEL_COLS), dtype=np.float32)])

y_mean = y_train.mean(axis=0, dtype=np.float64).astype(np.float32)
y_std = y_train.std(axis=0, dtype=np.float64).astype(np.float32)

if np.any(y_std < 1e-12):
    raise ValueError("Une cible possède une variance presque nulle.")

def norm_x(X): return ((X - x_mean) / x_std).astype(np.float32)
def norm_y(y): return ((y - y_mean) / y_std).astype(np.float32)
def denorm_y(y): return y * y_std + y_mean

Xn_train, Xn_val, Xn_test = norm_x(X_train), norm_x(X_val), norm_x(X_test)
yn_train, yn_val, yn_test = norm_y(y_train), norm_y(y_val), norm_y(y_test)

np.savez(LSTM_NS_SCALER_FILE, x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std,
         input_cols=np.array(LSTM_NS_MODEL_INPUT_COLS), symbolic_cols=np.array(SYMBOLIC_MODEL_COLS))

print("y_mean :", y_mean)
print("y_std  :", y_std)
print("Scalers :", LSTM_NS_SCALER_FILE)

y_mean : [ 2.9294287e+03 -2.7507319e-04 -1.3133448e-04]
y_std  : [1.0255436e+04 4.9867359e-04 2.0269521e-03]
Scalers : C:\Users\Admin\Desktop\Projet_Artemis2\models\EMS_LSTM_neurosymbolic_scalers.npz


Dataset et DataLoaders

In [35]:
# Cellule 10 : Dataset et DataLoaders
class WindowDatasetNS(Dataset):
    def __init__(self, X, y, aux):
        self.X = torch.from_numpy(X); self.y = torch.from_numpy(y); self.aux = torch.from_numpy(aux)
    def __len__(self): return len(self.X)
    def __getitem__(self, index): return (self.X[index], self.y[index], self.aux[index])

generator = torch.Generator()
generator.manual_seed(RANDOM_SEED)

train_loader = DataLoader(WindowDatasetNS(Xn_train, yn_train, aux_train), batch_size=LSTM_NS_BATCH_SIZE,
                           shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, generator=generator)
val_loader = DataLoader(WindowDatasetNS(Xn_val, yn_val, aux_val), batch_size=LSTM_NS_BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(WindowDatasetNS(Xn_test, yn_test, aux_test), batch_size=LSTM_NS_BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset))

7241 3608 3609


Modèle

In [36]:
# Cellule 12 : modele
class LSTMNeuroSymbolique(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                             batch_first=True, dropout=(dropout if num_layers > 1 else 0.0))
        self.head = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x):
        output, _ = self.lstm(x)
        return self.head(output[:, -1, :])

torch.manual_seed(RANDOM_SEED)
model = LSTMNeuroSymbolique(n_features=len(LSTM_NS_MODEL_INPUT_COLS), hidden_size=LSTM_NS_HIDDEN_SIZE,
                             num_layers=LSTM_NS_NUM_LAYERS, dropout=LSTM_NS_DROPOUT).to(DEVICE)
print(model)
print("Device :", next(model.parameters()).device)

LSTMNeuroSymbolique(
  (lstm): LSTM(11, 64, num_layers=2, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=3, bias=True)
  )
)
Device : cuda:0


Perte composite

In [37]:
# Cellule 14 : perte composite
required_weights = {"soc_bounds", "physics", "rules"}
missing_weights = required_weights - set(LSTM_NS_LOSS_WEIGHTS)
if missing_weights:
    raise ValueError(f"Poids absents : {missing_weights}")

y_mean_t = torch.tensor(y_mean, dtype=torch.float32, device=DEVICE)
y_std_t = torch.tensor(y_std, dtype=torch.float32, device=DEVICE)

def denorm_y_torch(values): return values * y_std_t + y_mean_t

def composite_loss(prediction_normalized, target_normalized, aux):
    data_loss = torch.mean((prediction_normalized - target_normalized) ** 2)
    prediction = denorm_y_torch(prediction_normalized)
    dsoc_eb, dsoc_pb = prediction[:, 1], prediction[:, 2]
    dsoc_eb_phys, dsoc_pb_phys = aux[:, 0], aux[:, 1]
    protected_discharge = aux[:, 2]
    soc_eb_anchor, soc_pb_anchor = aux[:, 3], aux[:, 4]

    soc_eb_pred = soc_eb_anchor + dsoc_eb
    soc_pb_pred = soc_pb_anchor + dsoc_pb

    eb_lower, eb_upper = SOC_EB_MIN - SOC_TOL, SOC_EB_MAX + SOC_TOL
    pb_lower, pb_upper = SOC_PB_MIN - SOC_TOL, SOC_PB_MAX + SOC_TOL

    bounds_eb = (torch.relu(eb_lower - soc_eb_pred) / y_std_t[1]) ** 2
    bounds_eb += (torch.relu(soc_eb_pred - eb_upper) / y_std_t[1]) ** 2
    bounds_pb = (torch.relu(pb_lower - soc_pb_pred) / y_std_t[2]) ** 2
    bounds_pb += (torch.relu(soc_pb_pred - pb_upper) / y_std_t[2]) ** 2
    bounds_loss = bounds_eb.mean() + bounds_pb.mean()

    physics_loss = torch.mean(((dsoc_eb - dsoc_eb_phys) / y_std_t[1]) ** 2)
    physics_loss += torch.mean(((dsoc_pb - dsoc_pb_phys) / y_std_t[2]) ** 2)

    forbidden_discharge = torch.relu(-dsoc_eb)
    rules_loss = torch.mean((protected_discharge * forbidden_discharge / y_std_t[1]) ** 2)

    total_loss = (data_loss + LSTM_NS_LOSS_WEIGHTS["soc_bounds"] * bounds_loss
                  + LSTM_NS_LOSS_WEIGHTS["physics"] * physics_loss
                  + LSTM_NS_LOSS_WEIGHTS["rules"] * rules_loss)

    components = {"data": data_loss.item(), "bounds": bounds_loss.item(),
                  "physics": physics_loss.item(), "rules_proxy": rules_loss.item()}
    return total_loss, components

In [38]:
# Cellule 12
class LSTMNeuroSymbolique(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=num_layers,
                             batch_first=True, dropout=(dropout if num_layers > 1 else 0.0))
        self.head = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x):
        output, _ = self.lstm(x)
        return self.head(output[:, -1, :])

torch.manual_seed(RANDOM_SEED)
model = LSTMNeuroSymbolique(n_features=len(LSTM_NS_MODEL_INPUT_COLS), hidden_size=LSTM_NS_HIDDEN_SIZE,
                             num_layers=LSTM_NS_NUM_LAYERS, dropout=LSTM_NS_DROPOUT).to(DEVICE)
print(model)
print("Device :", next(model.parameters()).device)

LSTMNeuroSymbolique(
  (lstm): LSTM(11, 64, num_layers=2, batch_first=True, dropout=0.2)
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=3, bias=True)
  )
)
Device : cuda:0


In [39]:
# Cellule 14
required_weights = {"soc_bounds", "physics", "rules"}
missing_weights = required_weights - set(LSTM_NS_LOSS_WEIGHTS)
if missing_weights:
    raise ValueError(f"Poids absents : {missing_weights}")

y_mean_t = torch.tensor(y_mean, dtype=torch.float32, device=DEVICE)
y_std_t = torch.tensor(y_std, dtype=torch.float32, device=DEVICE)

def denorm_y_torch(values): return values * y_std_t + y_mean_t

def composite_loss(prediction_normalized, target_normalized, aux):
    data_loss = torch.mean((prediction_normalized - target_normalized) ** 2)
    prediction = denorm_y_torch(prediction_normalized)
    dsoc_eb, dsoc_pb = prediction[:, 1], prediction[:, 2]
    dsoc_eb_phys, dsoc_pb_phys = aux[:, 0], aux[:, 1]
    protected_discharge = aux[:, 2]
    soc_eb_anchor, soc_pb_anchor = aux[:, 3], aux[:, 4]

    soc_eb_pred = soc_eb_anchor + dsoc_eb
    soc_pb_pred = soc_pb_anchor + dsoc_pb

    eb_lower, eb_upper = SOC_EB_MIN - SOC_TOL, SOC_EB_MAX + SOC_TOL
    pb_lower, pb_upper = SOC_PB_MIN - SOC_TOL, SOC_PB_MAX + SOC_TOL

    bounds_eb = (torch.relu(eb_lower - soc_eb_pred) / y_std_t[1]) ** 2
    bounds_eb += (torch.relu(soc_eb_pred - eb_upper) / y_std_t[1]) ** 2
    bounds_pb = (torch.relu(pb_lower - soc_pb_pred) / y_std_t[2]) ** 2
    bounds_pb += (torch.relu(soc_pb_pred - pb_upper) / y_std_t[2]) ** 2
    bounds_loss = bounds_eb.mean() + bounds_pb.mean()

    physics_loss = torch.mean(((dsoc_eb - dsoc_eb_phys) / y_std_t[1]) ** 2)
    physics_loss += torch.mean(((dsoc_pb - dsoc_pb_phys) / y_std_t[2]) ** 2)

    forbidden_discharge = torch.relu(-dsoc_eb)
    rules_loss = torch.mean((protected_discharge * forbidden_discharge / y_std_t[1]) ** 2)

    total_loss = (data_loss + LSTM_NS_LOSS_WEIGHTS["soc_bounds"] * bounds_loss
                  + LSTM_NS_LOSS_WEIGHTS["physics"] * physics_loss
                  + LSTM_NS_LOSS_WEIGHTS["rules"] * rules_loss)

    components = {"data": data_loss.item(), "bounds": bounds_loss.item(),
                  "physics": physics_loss.item(), "rules_proxy": rules_loss.item()}
    return total_loss, components

Entraînement

In [40]:
# Cellule 16 : entrainement
optimizer = torch.optim.Adam(model.parameters(), lr=LSTM_NS_LEARNING_RATE)
best_val, best_epoch, patience_count = float("inf"), -1, 0
history = {"train": [], "validation": []}
components_history = []

for epoch in range(LSTM_NS_EPOCHS):
    model.train()
    train_loss = 0.0
    component_sum = {"data": 0.0, "bounds": 0.0, "physics": 0.0, "rules_proxy": 0.0}

    for xb, yb, auxb in train_loader:
        xb, yb, auxb = xb.to(DEVICE), yb.to(DEVICE), auxb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(xb)
        loss, components = composite_loss(prediction, yb, auxb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
        for key in component_sum:
            component_sum[key] += components[key] * xb.size(0)

    train_loss /= len(train_loader.dataset)
    for key in component_sum:
        component_sum[key] /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb, auxb in val_loader:
            xb, yb, auxb = xb.to(DEVICE), yb.to(DEVICE), auxb.to(DEVICE)
            prediction = model(xb)
            loss, _ = composite_loss(prediction, yb, auxb)
            val_loss += loss.item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    if not np.isfinite(train_loss): raise ValueError("Train loss non finie.")
    if not np.isfinite(val_loss): raise ValueError("Validation loss non finie.")

    history["train"].append(train_loss)
    history["validation"].append(val_loss)
    components_history.append(component_sum.copy())

    if val_loss < best_val - 1e-6:
        best_val, best_epoch, patience_count = val_loss, epoch, 0
        torch.save(model.state_dict(), LSTM_NS_CHECKPOINT)
    else:
        patience_count += 1

    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:3d}/{LSTM_NS_EPOCHS} train={train_loss:.6f} val={val_loss:.6f} "
              f"data={component_sum['data']:.4f} bounds={component_sum['bounds']:.4f} "
              f"physics={component_sum['physics']:.4f} rules={component_sum['rules_proxy']:.4f} "
              f"patience={patience_count}")

    if patience_count >= LSTM_NS_PATIENCE:
        print(f"Arrêt anticipé à l'époque {epoch + 1}.")
        break

Epoch   1/100 train=0.894795 val=0.897835 data=0.7582 bounds=0.0007 physics=1.3650 rules=0.0000 patience=0
Epoch   5/100 train=0.584103 val=0.638120 data=0.5090 bounds=0.0017 physics=0.7497 rules=0.0000 patience=0
Epoch  10/100 train=0.364344 val=0.344131 data=0.3192 bounds=0.0024 physics=0.4493 rules=0.0000 patience=0
Epoch  15/100 train=0.213003 val=0.235519 data=0.1873 bounds=0.0022 physics=0.2553 rules=0.0000 patience=0
Epoch  20/100 train=0.142830 val=0.133334 data=0.1267 bounds=0.0025 physics=0.1585 rules=0.0000 patience=0
Epoch  25/100 train=0.104348 val=0.091154 data=0.0930 bounds=0.0025 physics=0.1108 rules=0.0000 patience=0
Epoch  30/100 train=0.081980 val=0.079377 data=0.0734 bounds=0.0022 physics=0.0833 rules=0.0000 patience=0
Epoch  35/100 train=0.068186 val=0.068446 data=0.0610 bounds=0.0021 physics=0.0702 rules=0.0000 patience=1
Epoch  40/100 train=0.061145 val=0.053561 data=0.0547 bounds=0.0012 physics=0.0631 rules=0.0000 patience=0
Epoch  45/100 train=0.055334 val=0.05

Chargement du meilleur modèle

In [41]:
# Cellule 18 : chargement du meilleur modele
if not LSTM_NS_CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint absent : {LSTM_NS_CHECKPOINT}")

try:
    state_dict = torch.load(LSTM_NS_CHECKPOINT, map_location=DEVICE, weights_only=True)
except TypeError:
    state_dict = torch.load(LSTM_NS_CHECKPOINT, map_location=DEVICE)

model.load_state_dict(state_dict)
print("Meilleure époque :", best_epoch + 1)
print("Meilleure val loss :", best_val)

Meilleure époque : 96
Meilleure val loss : 0.016247464771363924


 Prédictions et comparaison

In [42]:
# Cellule 20 : predictions et comparaison
def mae(y_true, y_pred): return float(np.mean(np.abs(y_true - y_pred)))
def rmse(y_true, y_pred): return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
def r2_score_np(y_true, y_pred):
    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(denominator, 0.0): return np.nan
    return float(1.0 - np.sum((y_true - y_pred) ** 2) / denominator)

def predict(model, loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for xb, _, _ in loader:
            xb = xb.to(DEVICE)
            output = model(xb)
            predictions.append(output.cpu().numpy())
    return np.concatenate(predictions, axis=0)

pred_test = denorm_y(predict(model, test_loader))
target_names = ["Pdem", "delta_SOC_EB", "delta_SOC_PB"]

simple_file = TABLES_DIR / "lstm_seul_vs_persistence.csv"
simple_metrics = pd.read_csv(simple_file).set_index("target")

rows = []
for index, target in enumerate(target_names):
    y_true, y_pred = y_test[:, index], pred_test[:, index]
    ns_mae = mae(y_true, y_pred)
    rows.append({
        "target": target, "lstm_ns_mae": ns_mae,
        "lstm_ns_rmse": rmse(y_true, y_pred), "lstm_ns_r2": r2_score_np(y_true, y_pred),
        "lstm_seul_mae": simple_metrics.loc[target, "lstm_mae"],
        "lstm_seul_rmse": simple_metrics.loc[target, "lstm_rmse"],
        "lstm_seul_r2": simple_metrics.loc[target, "lstm_r2"],
        "mae_improvement_percent": 100.0 * (simple_metrics.loc[target, "lstm_mae"] - ns_mae) / simple_metrics.loc[target, "lstm_mae"],
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)
comparison_df.to_csv(TABLES_DIR / "lstm_ns_vs_lstm_seul.csv", index=False)

,target,lstm_ns_mae,lstm_ns_rmse,lstm_ns_r2,lstm_seul_mae,lstm_seul_rmse,lstm_seul_r2,mae_improvement_percent
0,Pdem,3237.882080,7002.870117,0.582861,3506.524170,8126.951660,0.438197,7.661207
1,delta_SOC_EB,0.000093,0.000188,0.842324,0.000091,0.000170,0.870454,-2.212230
2,delta_SOC_PB,0.000971,0.002720,0.132018,0.000911,0.002563,0.229330,-6.618990


In [43]:
# Cellule 22
pred_df = meta_test.copy()
pred_df["Pdem_true"] = y_test[:, 0]; pred_df["Pdem_pred"] = pred_test[:, 0]
pred_df["delta_SOC_EB_true"] = y_test[:, 1]; pred_df["delta_SOC_EB_pred"] = pred_test[:, 1]
pred_df["delta_SOC_PB_true"] = y_test[:, 2]; pred_df["delta_SOC_PB_pred"] = pred_test[:, 2]
pred_df["SOC_EB_pred"] = pred_df["SOC_EB_anchor"] + pred_df["delta_SOC_EB_pred"]
pred_df["SOC_PB_pred"] = pred_df["SOC_PB_anchor"] + pred_df["delta_SOC_PB_pred"]

pred_df["SOC_EB_target_strict_out"] = (pred_df["SOC_EB_target"] < SOC_EB_MIN) | (pred_df["SOC_EB_target"] > SOC_EB_MAX)
pred_df["SOC_EB_pred_strict_out"] = (pred_df["SOC_EB_pred"] < SOC_EB_MIN) | (pred_df["SOC_EB_pred"] > SOC_EB_MAX)
pred_df["SOC_EB_pred_significant_out"] = (pred_df["SOC_EB_pred"] < SOC_EB_MIN - SOC_TOL) | (pred_df["SOC_EB_pred"] > SOC_EB_MAX + SOC_TOL)
pred_df["SOC_EB_model_induced_out"] = pred_df["SOC_EB_pred_strict_out"] & ~pred_df["SOC_EB_target_strict_out"]
pred_df["SOC_PB_pred_strict_out"] = (pred_df["SOC_PB_pred"] < SOC_PB_MIN) | (pred_df["SOC_PB_pred"] > SOC_PB_MAX)

prediction_file = PREDICTIONS_DIR / "predictions_lstm_neurosymbolique.csv"
pred_df.to_csv(prediction_file, index=False)

summary = pd.DataFrame([
    {"indicator": "SOC_EB cible hors bornes strictes", "count": int(pred_df["SOC_EB_target_strict_out"].sum())},
    {"indicator": "SOC_EB prédit hors bornes strictes", "count": int(pred_df["SOC_EB_pred_strict_out"].sum())},
    {"indicator": "SOC_EB prédit hors bornes significatives", "count": int(pred_df["SOC_EB_pred_significant_out"].sum())},
    {"indicator": "SOC_EB sorties induites par le modèle", "count": int(pred_df["SOC_EB_model_induced_out"].sum())},
    {"indicator": "SOC_PB prédit hors bornes", "count": int(pred_df["SOC_PB_pred_strict_out"].sum())},
])
display(summary)
summary.to_csv(TABLES_DIR / "lstm_ns_soc_violations.csv", index=False)
print("Prédictions :", prediction_file)
print("Checkpoint  :", LSTM_NS_CHECKPOINT)

,indicator,count
0,SOC_EB cible hors bornes strictes,0
1,SOC_EB prédit hors bornes strictes,276
2,SOC_EB prédit hors bornes significatives,109
3,SOC_EB sorties induites par le modèle,276
4,SOC_PB prédit hors bornes,0


Prédictions : C:\Users\Admin\Desktop\Projet_Artemis2\results\predictions\predictions_lstm_neurosymbolique.csv
Checkpoint  : C:\Users\Admin\Desktop\Projet_Artemis2\models\checkpoints\EMS_LSTM_neurosymbolic.pt


In [44]:
# Cellule 24
rule_active = aux_test[:, 2] > 0.5
true_delta_eb = y_test[:, 1]
pred_delta_eb = pred_test[:, 1]

print("Fenêtres où la règle est active :", int(rule_active.sum()))
print("Cibles réelles avec delta SOC_EB négatif :", int((rule_active & (true_delta_eb < 0)).sum()))
print("Prédictions avec delta SOC_EB négatif :", int((rule_active & (pred_delta_eb < 0)).sum()))
print("Taux de conformité strict du modèle :", 100.0 * np.mean(pred_delta_eb[rule_active] >= 0), "%")

Fenêtres où la règle est active : 374
Cibles réelles avec delta SOC_EB négatif : 0
Prédictions avec delta SOC_EB négatif : 205
Taux de conformité strict du modèle : 45.18716577540107 %


In [45]:
# Cellule 25
rule_diagnostic = pd.DataFrame({
    "SOC_EB_anchor": aux_test[:, 3], "rule_active": rule_active.astype(int),
    "delta_SOC_EB_true": true_delta_eb, "delta_SOC_EB_pred": pred_delta_eb,
})
rule_diagnostic = rule_diagnostic[rule_diagnostic["rule_active"] == 1].copy()
rule_diagnostic["true_rule_violation"] = rule_diagnostic["delta_SOC_EB_true"] < 0
rule_diagnostic["pred_rule_violation"] = rule_diagnostic["delta_SOC_EB_pred"] < 0
display(rule_diagnostic[["delta_SOC_EB_true", "delta_SOC_EB_pred", "true_rule_violation", "pred_rule_violation"]].describe())

,delta_SOC_EB_true,delta_SOC_EB_pred
count,374.000000,374.000000
mean,0.000018,-0.000107
std,0.000064,0.000429
min,0.000000,-0.001163
25%,0.000000,-0.000439
50%,0.000000,-0.000073
75%,0.000000,0.000255
max,0.000451,0.000635


In [46]:
# Cellule 20
def mae(y_true, y_pred): return float(np.mean(np.abs(y_true - y_pred)))
def rmse(y_true, y_pred): return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
def r2_score_np(y_true, y_pred):
    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    if np.isclose(denominator, 0.0): return np.nan
    return float(1.0 - np.sum((y_true - y_pred) ** 2) / denominator)

def predict(model, loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for xb, _, _ in loader:
            xb = xb.to(DEVICE)
            output = model(xb)
            predictions.append(output.cpu().numpy())
    return np.concatenate(predictions, axis=0)

pred_test = denorm_y(predict(model, test_loader))
target_names = ["Pdem", "delta_SOC_EB", "delta_SOC_PB"]

simple_file = TABLES_DIR / "lstm_seul_vs_persistence.csv"
simple_metrics = pd.read_csv(simple_file).set_index("target")

rows = []
for index, target in enumerate(target_names):
    y_true, y_pred = y_test[:, index], pred_test[:, index]
    ns_mae = mae(y_true, y_pred)
    rows.append({
        "target": target, "lstm_ns_mae": ns_mae,
        "lstm_ns_rmse": rmse(y_true, y_pred), "lstm_ns_r2": r2_score_np(y_true, y_pred),
        "lstm_seul_mae": simple_metrics.loc[target, "lstm_mae"],
        "lstm_seul_rmse": simple_metrics.loc[target, "lstm_rmse"],
        "lstm_seul_r2": simple_metrics.loc[target, "lstm_r2"],
        "mae_improvement_percent": 100.0 * (simple_metrics.loc[target, "lstm_mae"] - ns_mae) / simple_metrics.loc[target, "lstm_mae"],
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)
comparison_df.to_csv(TABLES_DIR / "lstm_ns_vs_lstm_seul.csv", index=False)

,target,lstm_ns_mae,lstm_ns_rmse,lstm_ns_r2,lstm_seul_mae,lstm_seul_rmse,lstm_seul_r2,mae_improvement_percent
0,Pdem,3237.882080,7002.870117,0.582861,3506.524170,8126.951660,0.438197,7.661207
1,delta_SOC_EB,0.000093,0.000188,0.842324,0.000091,0.000170,0.870454,-2.212230
2,delta_SOC_PB,0.000971,0.002720,0.132018,0.000911,0.002563,0.229330,-6.618990


In [47]:
# Cellule 26
rule_active_test = aux_test[:, 2] > 0.5
pred_test_raw = pred_test.copy()
pred_test_rule_safe = pred_test.copy()
pred_test_rule_safe[rule_active_test, 1] = np.maximum(pred_test_rule_safe[rule_active_test, 1], 0.0)

raw_violations = int((rule_active_test & (pred_test_raw[:, 1] < 0)).sum())
safe_violations = int((rule_active_test & (pred_test_rule_safe[:, 1] < 0)).sum())

print("Règle active :", int(rule_active_test.sum()))
print("Violations brutes :", raw_violations)
print("Violations après projection :", safe_violations)
print("Conformité brute :", 100.0 * np.mean(pred_test_raw[rule_active_test, 1] >= 0), "%")
print("Conformité après projection :", 100.0 * np.mean(pred_test_rule_safe[rule_active_test, 1] >= 0), "%")

Règle active : 374
Violations brutes : 205
Violations après projection : 0
Conformité brute : 45.18716577540107 %
Conformité après projection : 100.0 %


In [48]:
# Cellule 27
rule_summary = pd.DataFrame([
    {"model": "LSTM-NS brut", "active_windows": int(rule_active_test.sum()),
     "violations": int((rule_active_test & (pred_test_raw[:, 1] < 0)).sum()),
     "compliance_percent": float(100.0 * np.mean(pred_test_raw[rule_active_test, 1] >= 0))},
    {"model": "LSTM-NS + projection ontologique", "active_windows": int(rule_active_test.sum()),
     "violations": int((rule_active_test & (pred_test_rule_safe[:, 1] < 0)).sum()),
     "compliance_percent": float(100.0 * np.mean(pred_test_rule_safe[rule_active_test, 1] >= 0))},
])
display(rule_summary)
rule_summary.to_csv(TABLES_DIR / "lstm_ns_rule_compliance.csv", index=False)

,model,active_windows,violations,compliance_percent
0,LSTM-NS brut,374,205,45.187166
1,LSTM-NS + projection ontologique,374,0,100.000000


Analyse des violations

In [49]:

pred_df = meta_test.copy()

pred_df["Pdem_true"] = y_test[:, 0]
pred_df["Pdem_pred"] = pred_test[:, 0]

pred_df["delta_SOC_EB_true"] = y_test[:, 1]
pred_df["delta_SOC_EB_pred"] = pred_test[:, 1]

pred_df["delta_SOC_PB_true"] = y_test[:, 2]
pred_df["delta_SOC_PB_pred"] = pred_test[:, 2]

pred_df["SOC_EB_pred"] = (
    pred_df["SOC_EB_anchor"]
    + pred_df["delta_SOC_EB_pred"]
)

pred_df["SOC_PB_pred"] = (
    pred_df["SOC_PB_anchor"]
    + pred_df["delta_SOC_PB_pred"]
)

pred_df["SOC_EB_target_strict_out"] = (
    (pred_df["SOC_EB_target"] < SOC_EB_MIN)
    | (pred_df["SOC_EB_target"] > SOC_EB_MAX)
)

pred_df["SOC_EB_pred_strict_out"] = (
    (pred_df["SOC_EB_pred"] < SOC_EB_MIN)
    | (pred_df["SOC_EB_pred"] > SOC_EB_MAX)
)

pred_df["SOC_EB_pred_significant_out"] = (
    (
        pred_df["SOC_EB_pred"]
        < SOC_EB_MIN - SOC_TOL
    )
    | (
        pred_df["SOC_EB_pred"]
        > SOC_EB_MAX + SOC_TOL
    )
)

pred_df["SOC_EB_model_induced_out"] = (
    pred_df["SOC_EB_pred_strict_out"]
    & ~pred_df["SOC_EB_target_strict_out"]
)

pred_df["SOC_PB_pred_strict_out"] = (
    (pred_df["SOC_PB_pred"] < SOC_PB_MIN)
    | (pred_df["SOC_PB_pred"] > SOC_PB_MAX)
)

prediction_file = (
    PREDICTIONS_DIR
    / "predictions_lstm_neurosymbolique.csv"
)

pred_df.to_csv(
    prediction_file,
    index=False,
)

summary = pd.DataFrame([
    {
        "indicator": "SOC_EB cible hors bornes strictes",
        "count": int(
            pred_df[
                "SOC_EB_target_strict_out"
            ].sum()
        ),
    },
    {
        "indicator": "SOC_EB prédit hors bornes strictes",
        "count": int(
            pred_df[
                "SOC_EB_pred_strict_out"
            ].sum()
        ),
    },
    {
        "indicator": "SOC_EB prédit hors bornes significatives",
        "count": int(
            pred_df[
                "SOC_EB_pred_significant_out"
            ].sum()
        ),
    },
    {
        "indicator": "SOC_EB sorties induites par le modèle",
        "count": int(
            pred_df[
                "SOC_EB_model_induced_out"
            ].sum()
        ),
    },
    {
        "indicator": "SOC_PB prédit hors bornes",
        "count": int(
            pred_df[
                "SOC_PB_pred_strict_out"
            ].sum()
        ),
    },
])

display(summary)

summary.to_csv(
    TABLES_DIR
    / "lstm_ns_soc_violations.csv",
    index=False,
)

print("Prédictions :", prediction_file)
print("Checkpoint  :", LSTM_NS_CHECKPOINT)



,indicator,count
0,SOC_EB cible hors bornes strictes,0
1,SOC_EB prédit hors bornes strictes,276
2,SOC_EB prédit hors bornes significatives,109
3,SOC_EB sorties induites par le modèle,276
4,SOC_PB prédit hors bornes,0


Prédictions : C:\Users\Admin\Desktop\Projet_Artemis2\results\predictions\predictions_lstm_neurosymbolique.csv
Checkpoint  : C:\Users\Admin\Desktop\Projet_Artemis2\models\checkpoints\EMS_LSTM_neurosymbolic.pt


In [50]:
print(
    "Règle active train :",
    int(aux_train[:, 2].sum()),
    "/",
    len(aux_train),
)

print(
    "Règle active validation :",
    int(aux_val[:, 2].sum()),
    "/",
    len(aux_val),
)

print(
    "Règle active test :",
    int(aux_test[:, 2].sum()),
    "/",
    len(aux_test),
)

Règle active train : 0 / 7241
Règle active validation : 0 / 3608
Règle active test : 374 / 3609


In [51]:
rule_active = aux_test[:, 2] > 0.5

true_delta_eb = y_test[:, 1]
pred_delta_eb = pred_test[:, 1]

print("Fenêtres où la règle est active :", int(rule_active.sum()))

print(
    "Cibles réelles avec delta SOC_EB négatif :",
    int(
        (
            rule_active
            & (true_delta_eb < 0)
        ).sum()
    ),
)

print(
    "Prédictions avec delta SOC_EB négatif :",
    int(
        (
            rule_active
            & (pred_delta_eb < 0)
        ).sum()
    ),
)

print(
    "Taux de conformité strict du modèle :",
    100.0
    * np.mean(
        pred_delta_eb[rule_active] >= 0
    ),
    "%",
)

Fenêtres où la règle est active : 374
Cibles réelles avec delta SOC_EB négatif : 0
Prédictions avec delta SOC_EB négatif : 205
Taux de conformité strict du modèle : 45.18716577540107 %


In [52]:
rule_diagnostic = pd.DataFrame({
    "SOC_EB_anchor": aux_test[:, 3],
    "rule_active": rule_active.astype(int),
    "delta_SOC_EB_true": true_delta_eb,
    "delta_SOC_EB_pred": pred_delta_eb,
})

rule_diagnostic = rule_diagnostic[
    rule_diagnostic["rule_active"] == 1
].copy()

rule_diagnostic["true_rule_violation"] = (
    rule_diagnostic["delta_SOC_EB_true"] < 0
)

rule_diagnostic["pred_rule_violation"] = (
    rule_diagnostic["delta_SOC_EB_pred"] < 0
)

display(
    rule_diagnostic[
        [
            "delta_SOC_EB_true",
            "delta_SOC_EB_pred",
            "true_rule_violation",
            "pred_rule_violation",
        ]
    ].describe()
)

,delta_SOC_EB_true,delta_SOC_EB_pred
count,374.000000,374.000000
mean,0.000018,-0.000107
std,0.000064,0.000429
min,0.000000,-0.001163
25%,0.000000,-0.000439
50%,0.000000,-0.000073
75%,0.000000,0.000255
max,0.000451,0.000635


In [53]:
rule_active_test = aux_test[:, 2] > 0.5

pred_test_raw = pred_test.copy()
pred_test_rule_safe = pred_test.copy()

pred_test_rule_safe[
    rule_active_test,
    1,
] = np.maximum(
    pred_test_rule_safe[
        rule_active_test,
        1,
    ],
    0.0,
)

raw_violations = int(
    (
        rule_active_test
        & (pred_test_raw[:, 1] < 0)
    ).sum()
)

safe_violations = int(
    (
        rule_active_test
        & (pred_test_rule_safe[:, 1] < 0)
    ).sum()
)

print("Règle active :", int(rule_active_test.sum()))
print("Violations brutes :", raw_violations)
print("Violations après projection :", safe_violations)

print(
    "Conformité brute :",
    100.0
    * np.mean(
        pred_test_raw[
            rule_active_test,
            1,
        ] >= 0
    ),
    "%",
)

print(
    "Conformité après projection :",
    100.0
    * np.mean(
        pred_test_rule_safe[
            rule_active_test,
            1,
        ] >= 0
    ),
    "%",
)

Règle active : 374
Violations brutes : 205
Violations après projection : 0
Conformité brute : 45.18716577540107 %
Conformité après projection : 100.0 %


In [54]:
rule_summary = pd.DataFrame([
    {
        "model": "LSTM-NS brut",
        "active_windows": int(rule_active_test.sum()),
        "violations": int(
            (
                rule_active_test
                & (pred_test_raw[:, 1] < 0)
            ).sum()
        ),
        "compliance_percent": float(
            100.0
            * np.mean(
                pred_test_raw[
                    rule_active_test,
                    1,
                ] >= 0
            )
        ),
    },
    {
        "model": "LSTM-NS + projection ontologique",
        "active_windows": int(rule_active_test.sum()),
        "violations": int(
            (
                rule_active_test
                & (pred_test_rule_safe[:, 1] < 0)
            ).sum()
        ),
        "compliance_percent": float(
            100.0
            * np.mean(
                pred_test_rule_safe[
                    rule_active_test,
                    1,
                ] >= 0
            )
        ),
    },
])

display(rule_summary)

rule_summary.to_csv(
    TABLES_DIR / "lstm_ns_rule_compliance.csv",
    index=False,
)

,model,active_windows,violations,compliance_percent
0,LSTM-NS brut,374,205,45.187166
1,LSTM-NS + projection ontologique,374,0,100.000000
